# Features vs Labels

Explores relationships between **automatic features** (`features.csv`) and **manual annotation dimensions** across all three tasks (event relation, agency, setting).

Annotations are from annotator `tejo9855`, who has labeled all three tasks.  
Run `features/extract_features.py` first to generate `features.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
from scipy.stats import spearmanr, pearsonr

from nb_utils import setup_plots

plt = setup_plots()

## Load Data

In [ ]:
from nb_utils import load, annotator_labels, DIMENSIONS

FEATURES_CSV = '../features/features.csv'
ANNOTATOR    = 'tejo9855'


In [ ]:
# ── Automatic features ────────────────────────────────────────────────────────
feat_df = pd.read_csv(FEATURES_CSV)
print(f'Automatic features: {len(feat_df)} rows x {len(feat_df.columns)} cols')

# ── Source / topic metadata ────────────────────────────────────────────────────
corpus = load('corpus')
meta_df = pd.DataFrame({
    'safe_instance_id':     corpus['safe_instance_id'],
    'source':               corpus['dolma_source'],
    'topic_classification': corpus['topic_classification'],
    'narrative_label':      corpus['narrative_label'],
    'text_snippet':         corpus['sampled_text'].fillna('').map(
        lambda t: (t[:250] + '…') if len(t) > 250 else t),
})

feat_df = feat_df.merge(meta_df, on='safe_instance_id', how='left')
print(f'After metadata merge: {len(feat_df)} rows')

# ── Event relation annotations ─────────────────────────────────────────────────
er = load('event_relation')
er_df = pd.DataFrame({
    'safe_instance_id': er['safe_instance_id'],
    'er_span1_event':   er[f'span1_is_event_{ANNOTATOR}'].fillna(False).astype(bool).astype(int),
    'er_span2_event':   er[f'span2_is_event_{ANNOTATOR}'].fillna(False).astype(bool).astype(int),
    'er_temporal_order': er[f'temporal_order_{ANNOTATOR}'],
    'er_causality':      er[f'causality_rating_{ANNOTATOR}'],
})
# temporal order / causality are only meaningful when both spans are events
both = (er_df['er_span1_event'] == 1) & (er_df['er_span2_event'] == 1)
er_df.loc[~both, ['er_temporal_order', 'er_causality']] = None
er_df = er_df.dropna(subset=['er_span1_event', 'er_span2_event'], how='all')
print(f'Event relation: {len(er_df)} instances, '
      f'{er_df["er_temporal_order"].notna().sum()} with temporal order annotations')

# ── Agency annotations ─────────────────────────────────────────────────────────
ag_df = annotator_labels('agency', ANNOTATOR, dropna=False)
print(f'Agency: {len(ag_df)} instances')

# ── Setting annotations ────────────────────────────────────────────────────────
st_df = annotator_labels('setting', ANNOTATOR, dropna=False)
print(f'Setting: {len(st_df)} instances')

# ── Merged dataframes ──────────────────────────────────────────────────────────
full_df   = feat_df.copy()
ann_df    = feat_df.merge(ag_df, on='safe_instance_id') \
                   .merge(st_df, on='safe_instance_id') \
                   .merge(er_df, on='safe_instance_id')
er_full_df = feat_df.merge(er_df, on='safe_instance_id')

print(f'\nJoint annotation dataframe: {len(ann_df)} instances')
print(f'Event-relation + features:  {len(er_full_df)} instances')


In [ ]:
ann_df["text_snippet"]

In [ ]:
ann_df["agency_change_of_state"]

In [ ]:
feat_df_copy   = feat_df.copy()
setting_full_df    = feat_df_copy.merge(st_df, on='safe_instance_id')

## 1. Automatic vs Manual Concordance

Do the automatic proxies track what annotators actually judged?
- **Brysbaert concreteness** vs annotated `setting_concreteness`
- **Temporal mention rate** vs annotated `setting_temporal_grounding`

In [ ]:
import textwrap

def _wrap(text, width=55):
    return '<br>'.join(textwrap.wrap(str(text), width=width))

def plotly_scatter(df, x_col, y_col, xlabel, ylabel, title):
    sub = df[[x_col, y_col, 'safe_instance_id', 'source', 'text_snippet']].dropna(subset=[x_col, y_col]).copy()
    if len(sub) < 3:
        print(f'{title}: insufficient data (n={len(sub)})')
        return
    r_s, p_s = spearmanr(sub[x_col], sub[y_col])
    sub['_text_wrapped'] = sub['text_snippet'].apply(_wrap)
    fig = px.scatter(
        sub, x=x_col, y=y_col,
        trendline='ols',
        custom_data=['safe_instance_id', 'source', '_text_wrapped'],
        labels={x_col: xlabel, y_col: ylabel},
        title=f'{title}<br><sup>Spearman r = {r_s:.2f}, p = {p_s:.3f}  (n={len(sub)})</sup>',
        template='plotly_white',
    )
    fig.update_traces(
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'source: %{customdata[1]}<br>'
            f'{xlabel}: ' + '%{x:.3f}<br>'
            f'{ylabel}: ' + '%{y}<br>'
            '<br>%{customdata[2]}<extra></extra>'
        ),
        marker=dict(size=8, opacity=0.75),
        selector=dict(mode='markers'),
    )
    fig.update_layout(
        width=620, height=440,
        title_font_size=13,
        margin=dict(l=70, r=70, t=90, b=70),
        hoverlabel=dict(font_size=12, namelength=-1),
    )
    fig.show()

plotly_scatter(ann_df,
    'concreteness_mean', 'setting_concreteness',
    'Brysbaert concreteness (auto)', 'Setting concreteness (annotated)',
    'Concreteness: Automatic vs Manual')

plotly_scatter(ann_df,
    'temporal_mention_rate', 'setting_temporal_grounding',
    'Temporal mention rate (auto)', 'Temporal grounding (annotated)',
    'Temporal grounding: Automatic vs Manual')

## 2. Theory-Driven Correlations

### 2a. Sentiment × Agency Conflict & Emotion

In [ ]:
plotly_scatter(ann_df,
    'sentiment_compound', 'agency_conflict',
    'VADER compound sentiment', 'Agency conflict (annotated)',
    'Sentiment vs Conflict')

plotly_scatter(ann_df,
    'sentiment_compound', 'agency_emotion',
    'VADER compound sentiment', 'Agency emotion (annotated)',
    'Sentiment vs Emotion')

### 2b. Past Tense Verb Rate × Event Temporal Order

In [ ]:
seq_df = er_full_df[['past_tense_verb_rate', 'er_temporal_order']].dropna()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
if len(seq_df) >= 3:
    cat_order = ['span1_first', 'simultaneous', 'too_hard_to_tell']
    present   = [o for o in cat_order if o in seq_df['er_temporal_order'].values]
    sns.boxplot(data=seq_df, x='er_temporal_order', y='past_tense_verb_rate',
                order=present, palette='Set2', ax=ax)
    ax.set_xlabel('Temporal order (annotated)')
    ax.set_ylabel('Past-tense verb rate (auto)')
    ax.set_title('Past-tense verbs vs Temporal Order')
else:
    ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
            transform=ax.transAxes, color='#999', fontsize=11)
    ax.set_title('Past-tense verbs vs Temporal Order')
plt.tight_layout()
plt.show()
print(f'N = {len(seq_df)} instances with temporal order annotations')

### 2c. POV × Agency Dimensions (grouped boxplots)

In [ ]:
agency_dims = ['agency_focalization','agency_emotion','agency_cognition',
               'agency_change_of_state','agency_conflict']
dim_labels  = ['Focalization','Emotion','Cognition','Change of State','Conflict']

pov_df = ann_df[ann_df['pov_dominant'] != 'none'].copy()
pov_order = ['first', 'second', 'third']
pov_colors = {'first': '#4e9af1', 'second': '#f4a432', 'third': '#4caf50'}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes = axes.flatten()

for i, (dim, label) in enumerate(zip(agency_dims, dim_labels)):
    ax = axes[i]
    sub = pov_df[['pov_dominant', dim]].dropna()
    present_pov = [p for p in pov_order if p in sub['pov_dominant'].values]
    data_by_pov = [sub[sub['pov_dominant'] == p][dim].values for p in present_pov]
    bp = ax.boxplot(data_by_pov, labels=present_pov, patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    for patch, pov in zip(bp['boxes'], present_pov):
        patch.set_facecolor(pov_colors[pov])
        patch.set_alpha(0.7)
    for j, (pov, d) in enumerate(zip(present_pov, data_by_pov)):
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(d))
        ax.scatter(np.full(len(d), j+1) + jitter, d,
                   color=pov_colors[pov], alpha=0.5, s=25, zorder=3)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_ylabel('Rating (1–5)')
    ax.set_ylim(0.5, 5.5)
    counts = {p: (sub['pov_dominant']==p).sum() for p in present_pov}
    ax.set_xticklabels([f'{p}\n(n={counts[p]})' for p in present_pov], fontsize=9)

# hide the unused 6th panel
axes[5].set_visible(False)

patches = [mpatches.Patch(color=pov_colors[p], alpha=0.7, label=f'{p.capitalize()} person')
           for p in pov_order]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10, frameon=True,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Agency dimensions by dominant POV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Automatic × Manual Correlation Heatmap

Spearman correlations between all automatic features and all manual annotation dimensions.

In [ ]:
auto_cols = ['past_tense_verb_rate', 'sentiment_compound', 'sentiment_neg',
             'pov_first_rate', 'pov_second_rate', 'pov_third_rate',
             'concreteness_mean', 'temporal_mention_rate']
auto_labels = ['Past-tense verb rate', 'Sentiment (compound)', 'Sentiment (neg)',
               'POV: 1st person', 'POV: 2nd person', 'POV: 3rd person',
               'Concreteness (Brysbaert)', 'Temporal mention rate']

manual_cols = (['agency_focalization','agency_emotion','agency_cognition',
                'agency_change_of_state','agency_conflict'] +
               ['setting_concreteness','setting_temporal_grounding',
                'setting_spatial_grounding','setting_sensory'])
manual_labels = (['Focalization','Emotion','Cognition','Change of State','Conflict'] +
                 ['Setting: Concreteness','Setting: Temporal','Setting: Spatial','Setting: Sensory'])

corr_matrix = np.full((len(auto_cols), len(manual_cols)), np.nan)
pval_matrix = np.full((len(auto_cols), len(manual_cols)), np.nan)

for i, ac in enumerate(auto_cols):
    for j, mc in enumerate(manual_cols):
        sub = ann_df[[ac, mc]].dropna()
        if len(sub) >= 4:
            r, p = spearmanr(sub[ac], sub[mc])
            corr_matrix[i, j] = r
            pval_matrix[i, j] = p

corr_df = pd.DataFrame(corr_matrix, index=auto_labels, columns=manual_labels)

fig, ax = plt.subplots(figsize=(13, 6))
mask = np.isnan(corr_matrix)
sns.heatmap(corr_df, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.4, linecolor='#ddd', mask=mask,
            cbar_kws={'label': "Spearman's r", 'shrink': 0.8})

for i in range(len(auto_cols)):
    for j in range(len(manual_cols)):
        if not np.isnan(pval_matrix[i,j]) and pval_matrix[i,j] < 0.05:
            ax.text(j + 0.85, i + 0.2, '*', ha='center', va='center',
                    fontsize=10, color='black', fontweight='bold')

ax.set_title("Automatic features × Manual annotations  (Spearman's r, * p < .05)",
             fontsize=12, fontweight='bold', pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

ax.axvline(5, color='black', lw=1.5)
ax.axvline(9, color='black', lw=1.5)
ax.text(2.5, -0.6, 'Agency',  ha='center', fontsize=9, style='italic')
ax.text(7,   -0.6, 'Setting', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.show()

## 4. POV Groups × All Annotation Dimensions

Violin plots for every annotation dimension split by dominant POV.

In [ ]:
all_dims   = (['agency_focalization','agency_emotion','agency_cognition',
               'agency_change_of_state','agency_conflict'] +
              ['setting_concreteness','setting_temporal_grounding',
               'setting_spatial_grounding','setting_sensory'])
all_labels = (['Focalization','Emotion','Cognition','Change of State','Conflict'] +
              ['Concreteness','Temporal','Spatial','Sensory'])

pov_df = ann_df[ann_df['pov_dominant'] != 'none'].copy()
n_dims = len(all_dims)
ncols = 5
nrows = -(-n_dims // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.2, nrows*3.6), sharey=False)
axes = axes.flatten()

for i, (dim, label) in enumerate(zip(all_dims, all_labels)):
    ax = axes[i]
    sub = pov_df[['pov_dominant', dim]].dropna()
    present = [p for p in pov_order if p in sub['pov_dominant'].values]
    if len(sub) >= 3 and len(present) >= 2:
        sns.violinplot(data=sub, x='pov_dominant', y=dim, order=present,
                       palette=pov_colors, inner='box', ax=ax,
                       cut=0, density_norm='width')
        ax.set_xticklabels([f'{p}\n(n={(sub["pov_dominant"]==p).sum()})' for p in present],
                           fontsize=8)
    else:
        ax.text(0.5, 0.5, 'n too small', ha='center', va='center', transform=ax.transAxes,
                color='#999', fontsize=9)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_ylim(0.5, 5.5)
    ax.set_xlabel('')
    ax.set_ylabel('Rating (1–5)', fontsize=8)

for ax in axes[n_dims:]:
    ax.set_visible(False)

fig.suptitle('All annotation dimensions by dominant POV  (annotator: tejo9855)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Automatic Feature Distributions by Source & Topic

Do texts from different sources or topics differ systematically in automatic features?

In [ ]:
# ── By source ─────────────────────────────────────────────────────────────────
plot_features = ['sentiment_compound', 'past_tense_verb_rate',
                 'concreteness_mean', 'temporal_mention_rate',
                 'pov_third_rate', 'pov_first_rate']
feat_labels   = ['Sentiment (compound)', 'Past-tense verb rate',
                 'Concreteness', 'Temporal mention rate',
                 'POV: 3rd person rate', 'POV: 1st person rate']

src_counts = full_df['source'].value_counts()
top_sources = src_counts[src_counts >= 10].index.tolist()
src_df = full_df[full_df['source'].isin(top_sources)].copy()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, (feat, flabel) in enumerate(zip(plot_features, feat_labels)):
    ax = axes[i]
    order = (src_df.groupby('source')[feat].median()
                   .sort_values(ascending=False).index.tolist())
    sns.boxplot(data=src_df, x='source', y=feat, order=order,
                palette='Set2', ax=ax, showfliers=False)
    ax.set_title(flabel, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('')

fig.suptitle('Automatic feature distributions by source  (boxes: IQR, line: median)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── By topic ──────────────────────────────────────────────────────────────────
topic_counts = full_df['topic_classification'].value_counts()
top_topics   = topic_counts[topic_counts >= 8].index.tolist()
top_df       = full_df[full_df['topic_classification'].isin(top_topics)].copy()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, (feat, flabel) in enumerate(zip(plot_features, feat_labels)):
    ax = axes[i]
    order = (top_df.groupby('topic_classification')[feat].median()
                   .sort_values(ascending=False).index.tolist())
    sns.boxplot(data=top_df, x='topic_classification', y=feat, order=order,
                palette='tab10', ax=ax, showfliers=False)
    ax.set_title(flabel, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=7)
    ax.set_ylabel('')

fig.suptitle('Automatic feature distributions by topic  (boxes: IQR, line: median)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()